## Data Exploration
**Package:** `conditional_embedding_model`
**Source notebook:** `train/TrainNS_data.ipynb`
**Purpose:** Load and inspect the CoopGrasping dataset, demonstrate the dataloader pipeline and negative sampling scheme.

## 1. Setup & Imports
All domain logic (dataset loading, negative sampling, DataLoader construction) comes from
`conditional_embedding_model.data`. Standard library and matplotlib are used for inspection
and visualization only.

In [ ]:
import os
import sys
import pickle
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Add the project root to sys.path so the package is importable
sys.path.insert(0, os.path.abspath(".."))

from conditional_embedding_model.data import (
    load_data_grasp,
    extract_dataset,
    get_negatives,
    batchify_wrapper,
    RandomGenerator,
    GraspDataset,
    divide_dataset,
)

CONFIG = {
    "data_root":       "../config/data",
    "dataset_file":    "CoopGrasping-v6_dataset.pkl",
    "batch_size":      64,
    "max_length":      100,
    "multiplier_factor": 1.1,
    "num_workers":     0,
    "seed":            42,
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print("CONFIG:", CONFIG)

## 2. Dataset Overview
Load the raw pickle file and inspect one `Dataset` situation object. Each situation holds:
- `map` – occupancy grid (90×90)
- `object_footprint` – object geometry (4×2)
- `grasping_approach` – per-grasp approach descriptors (150×4)
- `combinations` – (N, 3) matrix where column 2 encodes success (1) or failure (0)

In [ ]:
dataset_path = os.path.join(CONFIG["data_root"], CONFIG["dataset_file"])
with open(dataset_path, "rb") as f:
    raw_dataset = list(pickle.load(f).values())

print(f"Total situations in dataset: {len(raw_dataset)}")
print()

s = raw_dataset[0]
print(f"First situation attributes:")
print(f"  map shape              : {s.map.shape}   dtype={s.map.dtype}")
print(f"  object_footprint shape : {s.object_footprint.shape}  dtype={s.object_footprint.dtype}")
print(f"  grasping_approach shape: {s.grasping_approach.shape}  dtype={s.grasping_approach.dtype}")
print(f"  combinations shape     : {s.combinations.shape}  dtype={s.combinations.dtype}")
print()

# Class balance across all situations
n_pos = sum(int((sit.combinations[:, 2] == 1).sum()) for sit in raw_dataset)
n_neg = sum(int((sit.combinations[:, 2] == 0).sum()) for sit in raw_dataset)
total = n_pos + n_neg
print(f"Positive combinations : {n_pos:>7d}  ({100*n_pos/total:.1f}%)")
print(f"Negative combinations : {n_neg:>7d}  ({100*n_neg/total:.1f}%)")
print(f"Class balance (pos ratio): {n_pos/total:.4f}")

## 3. Dataloader Instantiation
`load_data_grasp` runs the full pipeline: unpack the pickle, extract center→context pairs
from every situation, sample negatives, split train/val/test (70/20/10), and return
`DataLoader` instances for train and validation plus a raw `GraspDataset` for the test
split. The feature structure dict encodes how to reconstruct each feature modality from
the flat concatenated feature vector.

In [ ]:
train_dl, val_dl, test_ds, feature_struct = load_data_grasp(
    CONFIG["batch_size"],
    CONFIG["multiplier_factor"],
    dataset_path,
    CONFIG["max_length"],
)

print(f"Train batches  : {len(train_dl)}")
print(f"Val batches    : {len(val_dl)}")
print(f"Test samples   : {len(test_ds)}")
print()
print("Feature structure (flat sizes and original shapes):")
for k in feature_struct["features_structured"]:
    flat  = feature_struct["features_structured"][k]
    shape = feature_struct["features_shape"][k]
    print(f"  {k:<22s}  flat_size={flat:>5d}   original_shape={shape}")

## 4. Negative Sampling Demo
`get_negatives` samples robot indices that are *not* in the context set for each center,
weighted by center-frequency raised to the 0.75 power (word2vec convention). The
`total_length` per row equals `K × max_context_length`, so every center gets the same
padded length in the collated batch.

In [ ]:
# Use a small slice so this cell runs quickly
with open(dataset_path, "rb") as f:
    raw_slice = list(pickle.load(f).values())[:30]

centers_s, contexts_s, features_s, bad_s, fst_s = extract_dataset(raw_slice)
print(f"Situations used : 30  (bad quality: {bad_s})")
print(f"Centers extracted: {len(centers_s)}")
max_ctx = max(len(c) for c in contexts_s)
min_ctx = min(len(c) for c in contexts_s)
print(f"Context length  : min={min_ctx}  max={max_ctx}")
print()

all_neg = get_negatives(contexts_s, centers_s, K=CONFIG["multiplier_factor"])

# %%
# Show positive vs negative counts for the first 10 centers
print(f"{'idx':>4}  {'#pos (context)':>14}  {'#neg (sampled)':>14}  {'total':>6}")
print("-" * 44)
for i in range(min(10, len(centers_s))):
    np_ = len(contexts_s[i])
    nn_ = len(all_neg[i])
    print(f"{i:>4}  {np_:>14}  {nn_:>14}  {np_+nn_:>6}")

## 5. Batch Visualization
Iterate once over the training DataLoader and inspect the first batch. Expected shapes:

| Tensor | Shape |
|--------|-------|
| centers | `(B, 1)` |
| contexts_negatives | `(B, max_length)` |
| masks | `(B, max_length)` |
| labels | `(B, max_length)` |
| features | `(B, feature_dim)` |

Mask is 1 for real (context + sampled negative) positions and 0 for padding.
Labels is 1 for context (positive) positions and 0 otherwise.

In [ ]:
for batch in train_dl:
    centers_t, ctx_neg_t, masks_t, labels_t, features_t = batch
    break

print("Batch tensor shapes and value ranges:")
named = [
    ("centers",            centers_t),
    ("contexts_negatives", ctx_neg_t),
    ("masks",              masks_t),
    ("labels",             labels_t),
    ("features",           features_t),
]
for name, t in named:
    print(
        f"  {name:<22s}  shape={str(t.shape):<27s}"
        f"  dtype={str(t.dtype):<16s}"
        f"  min={t.float().min().item():>8.3f}"
        f"  max={t.float().max().item():>8.3f}"
    )

# %%
# Bar chart: number of positive labels per sample in this batch
fig, ax = plt.subplots(figsize=(9, 3))
pos_counts = labels_t.sum(dim=1).numpy()
ax.bar(range(len(pos_counts)), pos_counts, color="steelblue", alpha=0.75)
ax.set_title("Positive labels per sample in one batch", fontsize=12)
ax.set_xlabel("Sample index in batch")
ax.set_ylabel("# positive labels")
plt.tight_layout()
plt.show()
print(f"avg={pos_counts.mean():.2f}  max={pos_counts.max()}  min={pos_counts.min()}")